# QC testing: Setting Hard QC thresholds

+ Plate 1: 96 samples(60/36), on MEGA kit v2, seq 15 sub-lib X 3 runs
+ Plate 2: 96 samples (88/8), on MEGA kit v2, seq 16 sub-lib X 3 runs
+ ~30 Billion reads per plate
+ Parse workflow has a low [doublet rate](https://support.parsebiosciences.com/hc/en-us/articles/360053107311-What-is-the-expected-doublet-rate#:~:text=Doublet%20rates%20are%20low%2C%20less,through%20the%20Whole%20Transcriptome%20workflow.): ~3% per 100K cells

***

Testing different filtering thresholds:

- Filter 1: Hard threshold for genes
    - min_genes_per_cell=500, 
    - max_genes_per_cell=6000,
    - min_counts_per_cell=500, 
    - min_cells_per_gene=5, 
    - min_cells_per_sample=10
    - Mito - 5%
    - Ribo - 5%
    - Filtering MT genes and MALAT1 removed
    - rm_doublets = False
    - rm_mad_outliers = False

- Filter 2: MAD thresholds per sample and doublet removal
    - Done in initial QC run
 
- Filter 3: Retain genes with at least `min_reads` in at least `min_samples` in the dataset
    - Discussed 100 samples, but indivdual plates have < 100 samples so applied n samples
 
***

In [ ]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
resolutions = [0.1, 0.2, 0.3, 0.4, 0.5]
batch_col = 'plate' # Should we set to plate and sample??
rm_doublets = False
rm_mad_outliers = False
run_filter_1 = True
run_filter_2 = False
run_filter_3 = False

# Load data

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, plate_path, scanpy_dir = initialize_env(plate)
logger.info("Loading data ...")

# Use the plate_path and scanpy_dir as needed
adata = load_and_dwnsmpl_data(None, plate_path)

In [ ]:
adata.obs

# QC metadata

In [ ]:
logger.info("Running QC ...")
adata.obs['sample'] = adata.obs['sample'].str.replace('sample_', '')
adata = adata[~adata.obs['sample'].str.endswith(tuple(['WGE', 'Hipp', 'Thal']))]
adata.obs['sublibrary'] = [x[1] for x in adata.obs.index.str.split('__s')] 
adata.obs['sample'].value_counts() # Cells per sample pre-filter

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

#plt.hist(adata.var['n_cells_by_counts'], bins=500)
#plt.xlabel('N cells expressing > 0')
#plt.ylabel('log(N genes)') # for visual clarity
#plt.axvline(2, color='red')
#plt.yscale('log') 

#sns.jointplot(
#   data=adata.obs,
#   x="log1p_total_counts",
#   y="log1p_n_genes_by_counts",
#   kind="hex",
#)
adata.obs

In [ ]:
adata.obs[['tscp_count', 'total_counts']] # Why the discrepancy? Duplicates? This is the filtered matrix??!!

In [ ]:
# Most expressed genes - Not such a big deal now?
logger.info("Most exp genes ...")
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
# Filter 1
# - Remove cells with < 300 genes OR < 500 reads
# - Remove genes expressed in < 5 cells
# - Remove samples with fewer than 500 cells

#### NOTE: tscp_count != total_counts. Scanpy uses total_counts for filters. Check why these values don't match ####

In [ ]:
if run_filter_1 is True:
    logger.info("Applying filter 1 ...")

    # Create the violin plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sc.pl.violin(
        adata,
        keys='gene_count',
        jitter=0.4,
        groupby='sample',
        rotation=90,
        size=1.5,
        ax=ax,
        show=False, 
        color='Red'
    )
    plt.title('Hard thresholds for gene per cell')
    plt.axhline(y = 300, color = 'r', linestyle = '-')
    plt.axhline(y = 6000, color = 'r', linestyle = '-') 
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sc.pl.violin(
        adata,
        keys='total_counts',
        jitter=0.4,
        groupby='sample',
        rotation=90,
        size=1.5,
        ax=ax,
        show=False, 
        color='Red'
    )
    plt.title('Hard thresholds for counts per cell')
    plt.axhline(y = 500, color = 'r', linestyle = '-')
    
    # 
    
    filter_anndata(
        adata, 
        min_genes_per_cell=500, 
        max_genes_per_cell=6000,
        min_counts_per_cell=500, 
        min_cells_per_gene=5, 
        min_cells_per_sample=10)
    adata.shape

In [ ]:
#Filter 2 
if run_filter_2 is True:
    logger.info("Detecting MAD outliers ...")
    detect_mad_outliers_per_sample(
        adata,
        group_column="sample",       # Column in `adata.obs` to group by
        target_column="total_counts",  # Column to detect outliers
        threshold=3,                # Number of MADs for outlier detection
        log=False,                  # Whether to log-transform the data
        use_median=True             # Use median and MAD (or mean and SD)
    )
    
    # Create the violin plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sc.pl.violin(
        adata,
        keys='total_counts',
        jitter=0.4,
        groupby='sample',
        rotation=90,
        size=1.5,
        ax=ax,
        show=False, 
        color='Red'
    )
    
    # Overlay outlier cells with red dots
    outliers = adata.obs.loc[adata.obs['mad_outlier']]
    ax.scatter(
        x=outliers['sample'],  # X-coordinates (groupby value)
        y=outliers['total_counts'],  # Y-coordinates (outlier values)
        color='red',
        label='Outliers',
        s=10,
        alpha=0.8
    )


In [ ]:
# Note this function includes mito and ribo gene removal so is always run
logger.info("Applying filter 2 ...") 
filter_cells_and_genes(adata, 5, 5, rm_doublets, rm_mad_outliers)

In [ ]:
# Filter 3
if run_filter_3 is True:
    sample_num = adata.obs['sample'].nunique() # Note that the maximum number of samples will be < 100 per plate
    def filter_genes_by_read_count(adata, min_reads=10, min_samples=100, inplace=True):
        """
        Identifies genes with at least `min_reads` in at least `min_samples` in the dataset.
        
        Parameters:
        - adata: AnnData object containing gene expression data.
        - min_reads: Minimum number of reads required per cell/sample.
        - min_samples: Minimum number of samples/cells that must meet the `min_reads` threshold.
        - inplace: If True, filters genes directly in `adata`. If False, returns a mask.
    
        Returns:
        - If `inplace=True`: Modifies `adata` by retaining only the filtered genes.
        - If `inplace=False`: Returns a boolean mask of the genes that meet the criteria.
        """
        # Count the number of samples with at least `min_reads` for each gene
        gene_mask = (adata.X >= min_reads).sum(axis=0).A1 >= min_samples
    
        if inplace:
            # Subset the AnnData object to retain only the filtered genes
            adata._inplace_subset_var(gene_mask)
        else:
            # Return the mask for external usage
            return gene_mask
    
    filter_genes_by_read_count(adata, min_reads=10, min_samples=100)
    print(f"Filtered genes: {adata.n_vars} remain.")
    
    #gene_mask = filter_genes_by_read_count(adata, min_reads=10, min_samples=sample_num, inplace=False)
    #print(f"Number of genes meeting criteria: {gene_mask.sum()}")
    
    # Create the violin plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sc.pl.violin(
        adata,
        keys='gene_count',
        jitter=0.4,
        groupby='sample',
        rotation=90,
        size=1.5,
        ax=ax,
        show=False, 
        color='Red'
    )
    plt.title('Hard thresholds for gene per cell')
    plt.axhline(y = 300, color = 'r', linestyle = '-')
    plt.axhline(y = 6000, color = 'r', linestyle = '-') 
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sc.pl.violin(
        adata,
        keys='total_counts',
        jitter=0.4,
        groupby='sample',
        rotation=90,
        size=1.5,
        ax=ax,
        show=False, 
        color='Red'
    )
    plt.title('Hard thresholds for counts per cell')
    plt.axhline(y = 500, color = 'r', linestyle = '-')


In [ ]:
# Plot UMAP
def run_default_scanpy(ann_obj):

    sc.pp.normalize_total(ann_obj) # Norm to median total count
    sc.pp.log1p(ann_obj)
    sc.pp.highly_variable_genes(ann_obj, n_top_genes=2000, flavor="seurat_v3")
    sc.tl.pca(ann_obj, svd_solver='arpack')
    sc.pp.neighbors(ann_obj)
    sc.tl.leiden(ann_obj)
    sc.pl.umap(ann_obj, color=['leiden'])

run_default_scanpy(adata)

In [ ]:
# Plot vlns
gene_sets = [
    ("general_genes", general_genes),
    ("pfc_features", pfc_features)
]

fig = plot_filtered_violin(
    adata, 
    gene_sets, 
    groupby_base="leiden", 
    resolutions=None, 
    clustering_algorithm="Leiden")
plt.show()  # Display the figure

In [ ]:
# Final Dimesnsions
adata.shape

In [ ]:
# Cells per sample after filter
adata.obs['sample'].value_counts()

In [ ]:
# Save
logger.info("Saving h5ad file ...")
adata.write(scanpy_dir + f'adata_qc_testing_{plate}.h5ad')
logger.info(f"{plate} qc run done.")